In [1]:
import json
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification
)
import evaluate

d:\Personal Projects\Corient Company Task\submission_nelson_wasekar\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "microsoft/deberta-v3-base"
DATA_FILE = r"D:\Personal Projects\Corient Company Task\submission_nelson_wasekar\src\component1_extraction\transactions_labeled.jsonl"
# DATA_FILE = "D:\Personal Projects\Corient Company Task\submission_nelson_wasekar\src\component1_extraction\transactions_labeled.jsonl"
OUTPUT_DIR = "./financial-ner-model"



In [3]:

LABELS = ["O", "B-ORG", "I-ORG", "B-AMOUNT", "I-AMOUNT",
          "B-DATE", "I-DATE", "B-REF", "I-REF",
          "B-METHOD", "I-METHOD"]

label2id = {x:i for i, x in enumerate(LABELS)}
id2label = {i:x for i, x in enumerate(LABELS)}

In [4]:

def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            rows.append(obj)
    return rows

In [5]:
def char_spans_to_bio(text, entities, offsets):
    labels = ["O"] * len(offsets)
    for start, end, entity_type in entities:
        first = True
        for i, (tok_start, tok_end) in enumerate(offsets):
            if tok_start == tok_end:
                continue
            # overlap between token and entity character span
            if tok_start < end and tok_end > start:
                prefix = "B-" if first else "I-"
                labels[i] = prefix + entity_type
                first = False
    return [label2id.get(x, label2id["O"]) for x in labels]

In [6]:
rows = load_jsonl(DATA_FILE)

# Arrow can't infer a type for mixed-content lists like [16, 42, "ORG"],
# so store entities as a JSON string and decode it later instead.
for r in rows:
    r["entities"] = json.dumps(r["entities"])

dataset = Dataset.from_list(rows)
split = dataset.train_test_split(test_size=0.2, seed=42)
train_ds, test_ds = split["train"], split["test"]
val_split = train_ds.train_test_split(test_size=0.125, seed=42)
train_ds, val_ds = val_split["train"], val_split["test"]

In [ ]:
# rows = load_jsonl(DATA_FILE)
# dataset = Dataset.from_list(rows)
# split = dataset.train_test_split(test_size=0.2, seed=42)
# train_ds, test_ds = split["train"], split["test"]
# val_split = train_ds.train_test_split(test_size=0.125, seed=42)
# train_ds, val_ds = val_split["train"], val_split["test"]   

In [8]:
%pip install sentencepiece protobuf tiktoken

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import sentencepiece
print("sentencepiece OK:", sentencepiece.__file__)

import google.protobuf
print("protobuf version:", google.protobuf.__version__)

import tiktoken
print("tiktoken OK:", tiktoken.__file__)

sentencepiece OK: d:\Personal Projects\Corient Company Task\submission_nelson_wasekar\.venv\Lib\site-packages\sentencepiece\__init__.py
protobuf version: 7.36.1
tiktoken OK: d:\Personal Projects\Corient Company Task\submission_nelson_wasekar\.venv\Lib\site-packages\tiktoken\__init__.py


In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,
        return_offsets_mapping=True
    )
    all_labels = []
    for i, offsets in enumerate(enc["offset_mapping"]):
        # all_labels.append(
        #     char_spans_to_bio(batch["text"][i], batch["entities"][i], offsets)
        # )
        all_labels.append(
                char_spans_to_bio(batch["text"][i], json.loads(batch["entities"][i]), offsets)
        )
    enc["labels"] = all_labels
    enc.pop("offset_mapping")
    return enc


In [11]:
train_tok = train_ds.map(tokenize_and_align, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tokenize_and_align, batched=True, remove_columns=val_ds.column_names)
test_tok = test_ds.map(tokenize_and_align, batched=True, remove_columns=test_ds.column_names)

Map: 100%|██████████| 100/100 [00:00<00:00, 4981.77 examples/s]


In [14]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id
)

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        pred_tags, lab_tags = [], []
        for p_id, l_id in zip(pred, lab):
            if l_id == -100:
                continue
            pred_tags.append(id2label[int(p_id)])
            lab_tags.append(id2label[int(l_id)])
        true_predictions.append(pred_tags)
        true_labels.append(lab_tags)

    result = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": result["overall_precision"],
        "recall": result["overall_recall"],
        "f1": result["overall_f1"],
        "accuracy": result["overall_accuracy"]
    }

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=20,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics
)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 10068.40it/s]
[transformers] DebertaV2ForTokenClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 

In [ ]:
trainer.train()
print("\nValidation metrics:")
print(trainer.evaluate())

print("\nTest metrics:")
print(trainer.evaluate(test_tok))

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
d:\Personal Projects\Corient Company Task\submission_nelson_wasekar\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,4.974683,nan,0.000000,0.000000,0.000000,0.274126
2,0.000000,nan,0.000000,0.000000,0.000000,0.274126


d:\Personal Projects\Corient Company Task\submission_nelson_wasekar\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
d:\Personal Projects\Corient Company Task\submission_nelson_wasekar\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.48s/it]
d:\Personal Projects\Corient Company Task\submission_nelson_wasekar\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be 

In [ ]:
import json, time

test_metrics = trainer.evaluate(test_tok)

# time inference on a few samples to get a rough p95 latency
import numpy as np
latencies = []
for text in test_ds["text"][:20]:
    start = time.perf_counter()
    tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    latencies.append((time.perf_counter() - start) * 1000)

per_class_f1 = {
    label: test_metrics.get(f"eval_{label}", {}).get("f1", None)
    for label in ["ORG", "AMOUNT", "DATE", "REF", "METHOD"]
}

results = {
    "approach": "fine_tuning",
    "entity_level_f1": test_metrics["eval_overall_f1"],
    "per_class_f1": per_class_f1,
    "inference_latency_p95_ms": float(np.percentile(latencies, 95)),
    "test_samples": len(test_ds),
    "model_used": MODEL_NAME,
}

with open("../outputs/eval_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(results)